In [10]:
# ==========================================
# AlphaPulse - Phase 1: Data Acquisition
# ==========================================

import yfinance as yf
import pandas as pd
import numpy as np
import sqlite3
import warnings

warnings.filterwarnings('ignore')

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [11]:
## 2. Portfolio Definition

# Define portfolio
stocks = ['AAPL', 'MSFT', 'GOOG', 'TSLA', 'AMZN']

start_date = "2020-01-01"
end_date = "2024-01-01"

print("Portfolio:", stocks)

Portfolio: ['AAPL', 'MSFT', 'GOOG', 'TSLA', 'AMZN']


In [12]:
## 3. Data Ingestion 

# Download data
raw_data = yf.download(stocks, start=start_date, end=end_date)

# Debug columns
print("Columns:\n", raw_data.columns)

# Extract Close safely
if isinstance(raw_data.columns, pd.MultiIndex):
    data = raw_data['Close']
else:
    data = raw_data[['Close']]

print("Data Extracted.")
data.head()

[*********************100%***********************]  5 of 5 completed


Columns:
 MultiIndex([( 'Close', 'AAPL'),
            ( 'Close', 'AMZN'),
            ( 'Close', 'GOOG'),
            ( 'Close', 'MSFT'),
            ( 'Close', 'TSLA'),
            (  'High', 'AAPL'),
            (  'High', 'AMZN'),
            (  'High', 'GOOG'),
            (  'High', 'MSFT'),
            (  'High', 'TSLA'),
            (   'Low', 'AAPL'),
            (   'Low', 'AMZN'),
            (   'Low', 'GOOG'),
            (   'Low', 'MSFT'),
            (   'Low', 'TSLA'),
            (  'Open', 'AAPL'),
            (  'Open', 'AMZN'),
            (  'Open', 'GOOG'),
            (  'Open', 'MSFT'),
            (  'Open', 'TSLA'),
            ('Volume', 'AAPL'),
            ('Volume', 'AMZN'),
            ('Volume', 'GOOG'),
            ('Volume', 'MSFT'),
            ('Volume', 'TSLA')],
           names=['Price', 'Ticker'])
Data Extracted.


Ticker,AAPL,AMZN,GOOG,MSFT,TSLA
Date,,,,,
2020-01-02,72.400513,94.900497,67.811768,152.158417,28.684000
2020-01-03,71.696655,93.748497,67.478989,150.263779,29.534000
2020-01-06,72.267937,95.143997,69.142845,150.652161,30.102667
2020-01-07,71.928047,95.343002,69.099693,149.278549,31.270666
2020-01-08,73.085121,94.598503,69.644226,151.656311,32.809334


In [14]:
## 4. Data Cleaning

# Check missing values
print("❗ Missing Values BEFORE:\n", data.isnull().sum())

# Forward fill
data = data.ffill()

# Drop remaining nulls
data = data.dropna()

print("Missing Values AFTER:\n", data.isnull().sum())

data.head()

❗ Missing Values BEFORE:
 Ticker
AAPL    0
AMZN    0
GOOG    0
MSFT    0
TSLA    0
dtype: int64
Missing Values AFTER:
 Ticker
AAPL    0
AMZN    0
GOOG    0
MSFT    0
TSLA    0
dtype: int64


Ticker,AAPL,AMZN,GOOG,MSFT,TSLA
Date,,,,,
2020-01-02,72.400513,94.900497,67.811768,152.158417,28.684000
2020-01-03,71.696655,93.748497,67.478989,150.263779,29.534000
2020-01-06,72.267937,95.143997,69.142845,150.652161,30.102667
2020-01-07,71.928047,95.343002,69.099693,149.278549,31.270666
2020-01-08,73.085121,94.598503,69.644226,151.656311,32.809334


In [15]:
## 5. Data Validation

print("Start Date:", data.index.min())
print("End Date:", data.index.max())

print("Duplicate Rows:", data.duplicated().sum())

print("\n Summary Stats:")
print(data.describe())

Start Date: 2020-01-02 00:00:00
End Date: 2023-12-29 00:00:00
Duplicate Rows: 0

 Summary Stats:
Ticker         AAPL         AMZN         GOOG         MSFT         TSLA
count   1006.000000  1006.000000  1006.000000  1006.000000  1006.000000
mean     137.957993   137.216247   107.655006   254.181750   209.126371
std       33.357830    27.468805    25.227777    54.810713    85.797682
min       54.213608    81.820000    52.400791   128.636337    24.081333
25%      120.397022   114.309002    87.560141   212.832901   160.210003
50%      142.816368   140.585007   109.746738   250.927361   223.489998
75%      163.195042   161.190620   130.387054   293.761879   262.967491
max      196.073090   186.570496   149.481766   376.219177   409.970001


In [16]:
## 6. Transform for SQL (VERY IMPORTANT)

# Reset index
data_reset = data.reset_index()

# Convert wide → long format
data_sql = data_reset.melt(
    id_vars=['Date'],
    var_name='Stock',
    value_name='Price'
)

print("SQL Format Ready.")
data_sql.head()

SQL Format Ready.


,Date,Stock,Price
0,2020-01-02,AAPL,72.400513
1,2020-01-03,AAPL,71.696655
2,2020-01-06,AAPL,72.267937
3,2020-01-07,AAPL,71.928047
4,2020-01-08,AAPL,73.085121


In [17]:
## 7. Save Clean Files

# Save datasets
data.to_csv("stock_prices_clean.csv")
data_sql.to_csv("stock_prices_sql.csv", index=False)

print("CSV Files Saved.")

CSV Files Saved.


In [18]:
## 8. Store in SQLite Database

# Connect to database
conn = sqlite3.connect("alphapulse.db")

# Store table
data_sql.to_sql(
    name="stock_prices",
    con=conn,
    if_exists="replace",
    index=False
)

# Verify
preview = pd.read_sql("SELECT * FROM stock_prices LIMIT 5;", conn)

conn.close()

print("Data stored in SQLite.")
preview 

Data stored in SQLite.


,Date,Stock,Price
0,2020-01-02 00:00:00,AAPL,72.400513
1,2020-01-03 00:00:00,AAPL,71.696655
2,2020-01-06 00:00:00,AAPL,72.267937
3,2020-01-07 00:00:00,AAPL,71.928047
4,2020-01-08 00:00:00,AAPL,73.085121


In [19]:
## 9. Final Summary

print("Phase 1 Completed Successfully!")

print("\n Dataset Info:")
print("Total Records:", len(data_sql))
print("Total Stocks:", data_sql['Stock'].nunique())
print("Date Range:", data_sql['Date'].min(), "to", data_sql['Date'].max())

Phase 1 Completed Successfully!

 Dataset Info:
Total Records: 5030
Total Stocks: 5
Date Range: 2020-01-02 00:00:00 to 2023-12-29 00:00:00
